# Ev tapşırığı — Geyim mağazası (Fashion-MNIST)
Bu notebook `Lesson6/torch_mnist.ipynb` dərsinin MLP/CNN yanaşması əsasında hazırlanıb. **Bütün nəticələr notebook işə salındıqda hesablanır; uydurma accuracy yoxdur.** İlk işə salınmada Fashion-MNIST datasetinin endirilməsi üçün internet lazımdır.

In [ ]:
import os, random, copy
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device)
CLASS_NAMES=['T-shirt/top','Trouser','Pullover','Dress','Coat','Sandal','Shirt','Sneaker','Bag','Ankle boot']
DATA_DIR='./data'


## 1. Datanı yükləyin
**Düşün:** MNIST-in 0.1307 və 0.3081 dəyərləri onun train şəkillərinin orta qiyməti və standart sapmasıdır. Fashion-MNIST başqa şəkillərdən ibarətdir, buna görə mean/std-ni yenidən train datasından hesablamalıyıq.

In [ ]:
raw_train=datasets.FashionMNIST(DATA_DIR,train=True,download=True)
pixels=raw_train.data.float()/255.0
MEAN=pixels.mean().item(); STD=pixels.std(unbiased=False).item()
print(f'Fashion-MNIST mean={MEAN:.6f}, std={STD:.6f}')
normal=transforms.Compose([transforms.ToTensor(),transforms.Normalize((MEAN,),(STD,))])
train_data=datasets.FashionMNIST(DATA_DIR,train=True,download=False,transform=normal)
test_data=datasets.FashionMNIST(DATA_DIR,train=False,download=True,transform=normal)
train_loader=DataLoader(train_data,batch_size=128,shuffle=True,num_workers=0)
test_loader=DataLoader(test_data,batch_size=256,shuffle=False,num_workers=0)
print('Train:',len(train_data),'Test:',len(test_data))

## 2. 16 şəkil
**Düşün (əvvəlcədən proqnoz):** Shirt–T-shirt/top, Pullover–Coat və Shirt–Coat bir-birinə bənzəyə bilər. Confusion matrix ilə bunu sonradan yoxlayacağıq.

In [ ]:
fig,axes=plt.subplots(4,4,figsize=(10,10))
for ax,i in zip(axes.flat,range(16)):
    ax.imshow(raw_train.data[i],cmap='gray');ax.set_title(CLASS_NAMES[int(raw_train.targets[i])]);ax.axis('off')
plt.tight_layout();plt.show()

## 3. MLP baseline (3 epoch)
**Düşün:** Fashion-MNIST-də geyim formaları bir-birinə oxşadığı üçün MNIST-in təxminən 97%-indən aşağı nəticə gözləyirəm. `train_one_epoch` və `evaluate` funksiyaları datadan asılı deyil, eyni loss/accuracy məntiqi ilə işləyir.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten=nn.Flatten();self.fc1=nn.Linear(784,128);self.fc2=nn.Linear(128,64);self.fc3=nn.Linear(64,10)
    def forward(self,x):
        x=self.flatten(x);x=torch.relu(self.fc1(x));x=torch.relu(self.fc2(x));return self.fc3(x)

def train_one_epoch(model,loader,optimizer,loss_fn):
    model.train();total=0.0
    for images,labels in loader:
        images,labels=images.to(device),labels.to(device)
        optimizer.zero_grad();loss=loss_fn(model(images),labels);loss.backward();optimizer.step();total+=loss.item()
    return total/len(loader)

def evaluate(model,loader):
    model.eval();correct=0
    with torch.no_grad():
        for images,labels in loader:
            pred=model(images.to(device)).argmax(1);correct+=(pred==labels.to(device)).sum().item()
    return correct/len(loader.dataset)

def fit(model,loader,epochs,lr=0.001):
    optimizer=torch.optim.Adam(model.parameters(),lr=lr);loss_fn=nn.CrossEntropyLoss()
    best_acc=-1;best_state=None
    for epoch in range(epochs):
        loss=train_one_epoch(model,loader,optimizer,loss_fn);acc=evaluate(model,test_loader)
        print(f'Epoch {epoch+1}/{epochs}: loss={loss:.4f}, test accuracy={acc*100:.2f}%')
        if acc>best_acc:best_acc=acc;best_state=copy.deepcopy(model.state_dict())
    return best_acc,best_state

mlp=MLP().to(device);mlp_acc,_=fit(mlp,train_loader,3)

## 4. CNN (5 epoch)
**Düşün:** Giriş yenə 1 kanallı 28×28 şəkildir və çıxış 10 sinifdir; buna görə dərsdəki `Net` ölçülərini dəyişmək lazım deyil. İki MaxPool nəticəsində 28→14→7 olur, `fc1` girişi `64×7×7`-dir.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(1,32,3,padding=1);self.conv2=nn.Conv2d(32,64,3,padding=1)
        self.pool=nn.MaxPool2d(2);self.fc1=nn.Linear(64*7*7,128);self.fc2=nn.Linear(128,10);self.dropout=nn.Dropout(.25)
    def forward(self,x):
        x=self.pool(torch.relu(self.conv1(x)));x=self.pool(torch.relu(self.conv2(x)))
        x=torch.flatten(x,1);return self.fc2(self.dropout(torch.relu(self.fc1(x))))

x,_=next(iter(train_loader));print('Input',tuple(x.shape))
with torch.no_grad():
    check=Net();a=check.pool(torch.relu(check.conv1(x)));print('After conv1+pool',tuple(a.shape))
    a=check.pool(torch.relu(check.conv2(a)));print('After conv2+pool',tuple(a.shape))
cnn=Net().to(device);cnn_acc,cnn_state=fit(cnn,train_loader,5)

## 5. Augmentation (5 epoch)
**Düşün:** Rəqəmi üfüqi çevirmək onun mənasını dəyişə bilər, geyim üçün isə çox vaxt uyğundur. Şaquli çevirmək real geyim görünüşünə uyğun deyil. Güclü RandomAffine mərkəzləşdirilmiş kiçik şəkilləri korlaya bilər; burada yalnız yüngül transformasiya tətbiq olunur.

In [ ]:
aug_transform=transforms.Compose([transforms.RandomHorizontalFlip(p=0.5),transforms.RandomAffine(degrees=5,translate=(0.05,0.05)),transforms.ToTensor(),transforms.Normalize((MEAN,),(STD,))])
aug_data=datasets.FashionMNIST(DATA_DIR,train=True,download=False,transform=aug_transform)
aug_loader=DataLoader(aug_data,batch_size=128,shuffle=True,num_workers=0)
cnn_aug=Net().to(device);aug_acc,aug_state=fit(cnn_aug,aug_loader,5)
print(f'CNN no aug: {cnn_acc*100:.2f}%; CNN aug: {aug_acc*100:.2f}%')

## 6. Səhvlərin analizi və confusion matrix
**Düşün:** Aşağıdakı kod ən çox qarışdırılan üç cütü və ən rahat tanınan sinfi **real proqnozlardan** çıxarır. 2-ci addımdakı proqnozla müqayisə edin. Səhv nümunələrdə 28×28 ölçü və oxşar siluet insan üçün də çətinlik yarada bilər.

In [ ]:
best_model=cnn if cnn_acc>=aug_acc else cnn_aug
best_acc=max(cnn_acc,aug_acc);best_state=cnn_state if cnn_acc>=aug_acc else aug_state
cm=torch.zeros(10,10,dtype=torch.int64);mistakes=[]
best_model.eval()
with torch.no_grad():
    for images,labels in test_loader:
        preds=best_model(images.to(device)).argmax(1).cpu()
        for i,(truth,pred) in enumerate(zip(labels,preds)):
            cm[int(truth),int(pred)]+=1
            if truth!=pred and len(mistakes)<16:mistakes.append((images[i,0].clone(),int(truth),int(pred)))
plt.figure(figsize=(10,8));plt.imshow(cm.numpy(),cmap='Blues');plt.colorbar()
plt.xticks(range(10),CLASS_NAMES,rotation=45,ha='right');plt.yticks(range(10),CLASS_NAMES)
plt.xlabel('Proqnoz');plt.ylabel('Həqiqi sinif');plt.title('Confusion matrix')
for i in range(10):
    for j in range(10):plt.text(j,i,str(int(cm[i,j])),ha='center',va='center',fontsize=7)
plt.tight_layout();plt.show()
pairs=sorted([(int(cm[i,j]+cm[j,i]),i,j) for i in range(10) for j in range(i+1,10)],reverse=True)[:3]
print('Ən çox qarışan 3 cüt:')
for count,i,j in pairs:print(CLASS_NAMES[i],'<->',CLASS_NAMES[j],':',count)
class_accuracy=cm.diag().float()/cm.sum(1).clamp(min=1)
print('Ən asan sinif:',CLASS_NAMES[int(class_accuracy.argmax())],f'({class_accuracy.max()*100:.2f}%)')
fig,axes=plt.subplots(4,4,figsize=(12,10))
for ax,item in zip(axes.flat,mistakes):
    img,truth,pred=item;ax.imshow(img*STD+MEAN,cmap='gray');ax.set_title(f'Əsl: {CLASS_NAMES[truth]}\nDedi: {CLASS_NAMES[pred]}',fontsize=9);ax.axis('off')
plt.tight_layout();plt.show()

## 7. Modeli saxlayın və yoxlayın
**Düşün:** `state_dict` yalnız modelin öyrənilmiş parametrlərini saxlayır, daha çevikdir. Yenidən yükləmək üçün eyni `Net` arxitekturasını yaratmaq və parametrləri ona ötürmək lazımdır.

In [ ]:
torch.save(best_state,'fashion_cnn.pt')
reloaded=Net().to(device)
reloaded.load_state_dict(torch.load('fashion_cnn.pt',map_location=device,weights_only=True))
reloaded_acc=evaluate(reloaded,test_loader)
print(f'Saxlanmazdan əvvəl: {best_acc*100:.2f}% | Yüklənəndən sonra: {reloaded_acc*100:.2f}%')
assert abs(reloaded_acc-best_acc)<1e-9
print('Model uğurla saxlanıldı və yükləndi.')

## Özünü yoxla — nəticə cədvəli

In [ ]:
print(f'{"Model":<22} {"Augmentation":<15} {"Epoch":<8} {"Test accuracy"}')
for name,aug,epoch,acc in [('MLP','yox',3,mlp_acc),('CNN','yox',5,cnn_acc),('CNN','bəli',5,aug_acc),('Ən yaxşı variant','-',5,best_acc)]:
    print(f'{name:<22} {aug:<15} {epoch:<8} {acc*100:.2f}%')
print('90% hədəfi:', 'ƏLDƏ EDİLDİ' if best_acc>=0.90 else 'HƏLƏ ƏLDƏ EDİLMƏDİ')

## Bonus A — nəticə 90%-dən aşağı olarsa
Üçüncü convolution layer və ya daha çox epoch sınaqdan keçirilə bilər. Üçüncü MaxPool-dan sonra 7×7 ölçü 3×3 olur (floor bölmə); `fc1` giriş ölçüsü buna uyğun yenilənməlidir. Train loss azalıb test accuracy artmırsa, bu **overfitting** əlaməti ola bilər. Aşağıdakı əlavə təlim əsas nəticələri dəyişmir.

In [ ]:
class ImprovedNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(nn.Conv2d(1,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(64,128,3,padding=1),nn.ReLU())
        self.classifier=nn.Sequential(nn.Flatten(),nn.Linear(128*7*7,128),nn.ReLU(),nn.Dropout(.35),nn.Linear(128,10))
    def forward(self,x):return self.classifier(self.features(x))

# İstəyə bağlı əlavə sınaq: aşağıdakı sətri True edin.
RUN_BONUS=False
if RUN_BONUS:
    improved=ImprovedNet().to(device)
    improved_acc,_=fit(improved,aug_loader,8)
    print('Bonus accuracy:',f'{improved_acc*100:.2f}%')